# CLAUD·IA — Primera traducción de voz

Este notebook utiliza SeamlessM4T v2 **antes del fine-tuning**. Recibe un fragmento de audio, traduce la frase y genera un nuevo audio. Mediremos el tiempo de generación para establecer nuestro punto de partida.

Primera prueba propuesta: graba « Gracias por venir » en un archivo WAV corto. La traducción de referencia al inglés es « Thank you for coming. ».

[Documentación de SeamlessM4T v2](https://huggingface.co/docs/transformers/model_doc/seamless_m4t_v2)

## 1. Instalar las dependencias

Ejecuta esta celda durante el primer uso y luego reinicia el kernel antes de continuar. Si utilizaste una versión anterior del notebook, esta celda reinstala las versiones compatibles de Transformers y Hugging Face Hub.

In [ ]:
%pip install "torch>=2.6" "transformers[sentencepiece]==5.22.1" "huggingface-hub>=1.32,<2" soundfile scipy

## 2. Elegir el audio y el idioma de destino

En Jupyter, coloca tu archivo WAV junto al notebook. En Colab, agrégalo desde el panel **Archivos**. Escribe su nombre a continuación y ejecuta la celda para escuchar el original.

Para la primera prueba, deja `tgt_lang = "eng"`. Las otras opciones son `fra` y `spa`.

In [ ]:
from math import gcd
from pathlib import Path

import soundfile as sf
from IPython.display import Audio, display
from scipy.signal import resample_poly

audio_path = Path("gracias_por_venir.wav")
tgt_lang = "eng"
input_rate = 16_000

if not audio_path.is_file():
    raise FileNotFoundError(f"Agrega el archivo WAV {audio_path.name} en la carpeta del notebook.")

audio, original_rate = sf.read(audio_path, dtype="float32", always_2d=True)
if len(audio) == 0:
    raise ValueError("El archivo de audio está vacío.")

# El promedio de los canales produce una señal mono.
mono = audio.mean(axis=1)
if original_rate != input_rate:
    divisor = gcd(original_rate, input_rate)
    mono = resample_poly(mono, input_rate // divisor, original_rate // divisor)

print(f"Audio original • {len(mono) / input_rate:.1f} s • mono a {input_rate} Hz")
display(Audio(mono, rate=input_rate))

## 3. Cargar el Transformer

La primera carga descarga varios gigabytes de parámetros. Ejecuta esta celda una sola vez por sesión. La CPU está seleccionada de forma predeterminada. En Colab, si dispones de GPU, puedes cambiar `use_gpu` a `True`.

In [ ]:
import torch
from transformers import AutoProcessor, SeamlessM4Tv2ForSpeechToSpeech

model_id = "facebook/seamless-m4t-v2-large"
use_gpu = False

if use_gpu and not torch.cuda.is_available():
    raise RuntimeError("Activa un entorno GPU o deja use_gpu en False.")

device = torch.device("cuda" if use_gpu else "cpu")
dtype = torch.float16 if use_gpu else torch.float32

processor = AutoProcessor.from_pretrained(model_id)
model = SeamlessM4Tv2ForSpeechToSpeech.from_pretrained(
    model_id, dtype=dtype, use_safetensors=True
).to(device)
model.eval()
print(f"Modelo cargado en {device}.")

## 4. Traducir y escuchar

El modelo produce una traducción de texto intermedia y genera el audio a partir de ella. El texto mostrado ayuda a comprobar la traducción durante el trabajo práctico. El audio resultante se puede escuchar justo debajo.

In [ ]:
from time import perf_counter

if tgt_lang not in {"eng", "fra", "spa"}:
    raise ValueError("Selecciona eng, fra o spa como idioma de destino.")

inputs = processor(audio=mono, sampling_rate=input_rate, return_tensors="pt").to(device)
inputs["input_features"] = inputs["input_features"].to(dtype=dtype)

if use_gpu:
    torch.cuda.synchronize()
start = perf_counter()
with torch.inference_mode():
    result = model.generate(**inputs, tgt_lang=tgt_lang, return_intermediate_token_ids=True)
if use_gpu:
    torch.cuda.synchronize()
elapsed = perf_counter() - start

translation = processor.batch_decode(
    result.sequences.cpu(), skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]
# Este notebook procesa una sola frase. La longitud puede ser un escalar.
length = result.waveform_lengths.item()
translated_audio = result.waveform.reshape(-1)[:length].float().cpu().numpy()

print(f"Traducción generada ({tgt_lang}) — {translation}")
print(f"Tiempo de generación — {elapsed:.2f} s, sin incluir la carga del modelo")
display(Audio(translated_audio, rate=model.config.sampling_rate))